# Chapter 5 · 금융 이벤트 ITS

**연구질문:** 사건 시점 전후 수준·추세 변화는 ITS로 회복되는가? (알려진 효과를 주입한 준합성 검증)

결과변수는 월별 지표(예: 변동성 지수), 처치 단위는 사건을 겪은 한 개의 시계열로 가정한 **합성** 설정이다.

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**잠재결과와 처치효과:** 처치 단위 $i$의 사후 효과 $\tau_{it}=Y_{it}(1)-Y_{it}(0)$. 관측되지 않는 $Y_{it}(0)$(반사실)을 어떻게 대체하느냐가 방법을 가른다.

**ITS(중단 시계열):** $Y_t=\beta_0+\beta_1 t+\beta_2 D_t+\beta_3 (t-T_0)D_t+\varepsilon_t$, $D_t=\mathbf 1[t\ge T_0]$. $\beta_2$=즉시 수준변화, $\beta_3$=기울기 변화. 식별은 "개입이 없었다면 사전추세가 이어졌다"와 **동시 사건 부재**에 의존한다. 오차의 자기상관 때문에 HAC(Newey–West) 표준오차를 쓴다.

**DID(이중차분):** $Y_{it}=\alpha_i+\lambda_t+\tau\, D_{it}+\varepsilon_{it}$. 식별은 **평행추세**(처치가 없었다면 처치집단과 비교집단의 평균 변화가 같다)에 의존한다. 사전 이벤트 스터디 계수가 0과 다르면 가정이 의심된다.

**합성통제(SCM):** 비처치 단위의 가중평균으로 처치 단위의 사전 경로를 재현한다. $\hat w=\arg\min_{w\ge0,\ \sum w=1}\sum_{t<T_0}\big(Y_{1t}-\sum_j w_j Y_{jt}\big)^2$, 효과 $\hat\tau_t=Y_{1t}-\sum_j \hat w_j Y_{jt}$ ($t\ge T_0$). 추론은 **in-space 위약**(각 도너를 가짜 처치로 놓고 사후/사전 RMSPE 비를 비교)으로 한다.

> 이 노트북의 합성 패널은 **효과 $\tau$를 우리가 알고 있다.** 추정기가 이를 회복하는지를 확인하는 것이 목적이며, 실제 사건의 효과 추정이 아니다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
# ---- 패널·인과추론용 최소제곱과 로버스트 표준오차 (수식이 코드에 그대로 보이도록 직접 구현) ----
def ols(y, X, names=None, cov="HC1", groups=None, maxlags=None):
    y = np.asarray(y, float).reshape(-1); X = np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    if cov == "nonrobust":
        V = XtX_inv * (e @ e) / (n - k)
    elif cov == "HC1":
        V = XtX_inv @ (X.T * e ** 2 @ X) @ XtX_inv * n / (n - k)
    elif cov == "HAC":                                   # Newey-West, Bartlett 커널
        L = maxlags if maxlags is not None else int(np.floor(4 * (n / 100) ** (2 / 9)))
        Xe = X * e[:, None]
        S = Xe.T @ Xe
        for l in range(1, L + 1):
            G = Xe[l:].T @ Xe[:-l]
            S += (1 - l / (L + 1)) * (G + G.T)
        V = XtX_inv @ S @ XtX_inv * n / (n - k)
    elif cov == "cluster":                               # 군집 로버스트 (소표본 보정 포함)
        g = pd.factorize(np.asarray(groups))[0]; G = g.max() + 1
        S = np.zeros((k, k))
        for j in range(G):
            s = (X[g == j] * e[g == j][:, None]).sum(axis=0)
            S += np.outer(s, s)
        V = XtX_inv @ S @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    else:
        raise ValueError(cov)
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    t = b / np.where(se > 0, se, np.nan)
    dof = (len(np.unique(groups)) - 1) if cov == "cluster" else (n - k)
    p = 2 * (1 - stats.t.cdf(np.abs(t), dof))
    names = names or [f"x{i}" for i in range(k)]
    return {"beta": b, "se": se, "t": t, "p": p, "resid": e, "V": V, "names": list(names), "n": n,
            "table": pd.DataFrame({"coef": b, "se": se, "t": t, "p": p}, index=names)}

def make_panel(N=24, T=48, start="2018-01-01", freq="MS", n_treated=3, T0=30, tau=-8.0, seed=SEED,
               effect_shape="step", parallel=False):
    # NOT real data: 요인모형 패널. 처치효과 tau를 알고 있으므로 추정기가 이를 회복하는지 확인할 수 있다.
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=T, freq=freq)
    f1 = np.cumsum(rng.normal(0, 0.5, T)); f2 = np.sin(np.arange(T) * 2 * np.pi / 12)
    lam1 = rng.normal(1, 0.3, N); lam2 = rng.normal(0, 1, N); alpha = rng.normal(50, 8, N)
    if parallel:                                   # 모든 단위가 같은 공통추세만 가짐 -> 평행추세가 성립하는 설계
        lam1 = np.ones(N); lam2 = np.zeros(N)
    Y = alpha[:, None] + lam1[:, None] * f1[None] + 2 * lam2[:, None] * f2[None] + rng.normal(0, 1.0, (N, T))
    treated = list(range(n_treated))
    for i in treated:
        post = np.arange(T) >= T0
        ramp = np.clip((np.arange(T) - T0 + 1) / 6, 0, 1) if effect_shape == "ramp" else 1.0
        Y[i] += tau * post * ramp
    df = pd.DataFrame(Y.T, index=dates, columns=[f"U{i:02d}" for i in range(N)])
    return df, [f"U{i:02d}" for i in treated], T0

## 2. 데이터

In [ ]:
TAU = -6.0
panel, treated, T0 = make_panel(N=24, T=48, n_treated=1, T0=30, tau=TAU, seed=SEED, parallel=True)   # 평행추세가 성립하도록 설계
DATA_KIND = f"SYNTHETIC (NOT real data): 공통추세 패널(평행추세 성립), 주입 효과 tau={TAU}"

print("DATA_KIND:", DATA_KIND)
print("패널 크기(시점×단위):", panel.shape, "| 처치 단위:", treated, "| 처치 시작 인덱스 T0 =", T0)
panel.iloc[:, :6].plot(figsize=(10, 3.6), title="일부 단위의 결과변수"); plt.axvline(panel.index[T0], color="k", ls="--"); plt.show()

## 3. ITS

In [ ]:
# ---- (1) ITS: 처치 단위 한 개의 시계열 ----
u = treated[0]; ys = panel[u].to_numpy(float); n = len(ys); t = np.arange(n)
D_ = (t >= T0).astype(float); TA = (t - T0) * D_
X = np.column_stack([np.ones(n), t, D_, TA])
fit_its = ols(ys, X, ["const", "trend", "level_change(beta2)", "slope_change(beta3)"], cov="HAC", maxlags=3)
show(fit_its["table"])
pre = np.arange(n) < T0
fit_pre = ols(ys[pre], np.column_stack([np.ones(pre.sum()), t[pre]]), ["const", "trend"], cov="nonrobust")
cf = fit_pre["beta"][0] + fit_pre["beta"][1] * t                          # 사전추세 외삽(반사실)
plt.figure(figsize=(10, 3.8)); plt.plot(panel.index, ys, label=f"observed ({u})"); plt.plot(panel.index, cf, "--", label="사전추세 외삽(반사실)")
plt.axvline(panel.index[T0], color="k", ls=":"); plt.legend(); plt.title("ITS"); plt.show()
dw = np.sum(np.diff(fit_its["resid"]) ** 2) / np.sum(fit_its["resid"] ** 2)
print(f"Durbin-Watson={dw:.2f} (2 근처면 1차 자기상관 약함; HAC로 표준오차 보정)")
its_avg_effect = float(np.mean(ys[T0:] - cf[T0:]))
print(f"사후 평균 (관측-외삽) = {its_avg_effect:.3f}" + (f" | 주입 효과 tau={TAU}" if TAU is not None else ""))

## 4. DID와 이벤트 스터디

In [ ]:
# ---- (2) DID: 처치 단위 vs 비교 단위, 이원고정효과 + 이벤트 스터디 ----
long = panel.reset_index().melt(id_vars=panel.index.name or "index", var_name="unit", value_name="y")
tcol = panel.index.name or "index"
long["t"] = long[tcol].map({v: i for i, v in enumerate(panel.index)}); long = long.drop(columns=tcol)
long["treated_unit"] = long["unit"].isin(treated).astype(float); long["post"] = (long["t"] >= T0).astype(float)
long["D"] = long["treated_unit"] * long["post"]

def twfe_design(df, regs):
    U = pd.get_dummies(df["unit"], drop_first=True, dtype=float); Tm = pd.get_dummies(df["t"], drop_first=True, dtype=float, prefix="t")
    cols = [np.ones(len(df))] + [df[r].to_numpy(float) for r in regs]
    names = ["const"] + list(regs)
    return np.column_stack(cols + [U.to_numpy(), Tm.to_numpy()]), names + list(U.columns) + list(Tm.columns)

Xd, nm = twfe_design(long, ["D"])
rows_ = []
for cv in ("HC1", "cluster"):
    f_ = ols(long["y"], Xd, nm, cov=cv, groups=long["unit"])
    rows_.append({"SE 종류": cv, "tau_hat": float(f_["beta"][1]), "se": float(f_["se"][1]), "p": float(f_["p"][1])})
fit_did = ols(long["y"], Xd, nm, cov="cluster", groups=long["unit"])
print("TWFE DID | 처치 단위 수 =", len(treated), "| 비교 단위 수 =", panel.shape[1] - len(treated))
show(pd.DataFrame(rows_).set_index("SE 종류"))
if len(treated) < 5:
    print("주의: 처치 군집이 적으면 군집 SE는 과소추정되기 쉽다. 아래 순열(위약) p값을 더 신뢰한다. 실제 연구는 wild cluster bootstrap 등을 사용한다.")

# 순열 추론: 비교 단위에서 같은 수의 가짜 처치 단위를 뽑아 DID 추정치의 분포를 만든다
Pv = panel.to_numpy(float); ctrl_idx = [i for i, c in enumerate(panel.columns) if c not in treated]
def did_2x2(idx_t, idx_c):
    f = lambda idx, sl: Pv[sl][:, idx].mean()
    return (f(idx_t, slice(T0, None)) - f(idx_t, slice(0, T0))) - (f(idx_c, slice(T0, None)) - f(idx_c, slice(0, T0)))
rng_ = np.random.default_rng(SEED); draws = []
for _ in range(500):
    pick = rng_.choice(ctrl_idx, size=len(treated), replace=False); rest = [i for i in ctrl_idx if i not in pick]
    draws.append(did_2x2(list(pick), rest))
tau_2x2 = did_2x2([i for i, c in enumerate(panel.columns) if c in treated], ctrl_idx)
p_perm_did = (1 + np.sum(np.abs(draws) >= abs(tau_2x2))) / (1 + len(draws))
print(f"2x2 DID 추정치 = {tau_2x2:.3f} | 위약 분포 sd = {np.std(draws):.3f} | 순열 p = {p_perm_did:.3f}")

# 이벤트 스터디: 처치 시점 대비 상대시점 더미(-1 기준) -> 사전 계수가 0 근처인지 평행추세 점검
long["rel"] = long["t"] - T0
rels_all = [r for r in sorted(long["rel"].unique()) if r != -1]          # 모든 상대시점 더미(-1 기준): 끝점을 묶지 않아 편의가 생기지 않게 함
for r in rels_all:
    long[f"ev_{r}"] = ((long["rel"] == r) & (long["treated_unit"] == 1)).astype(float)
Xe, nme = twfe_design(long, [f"ev_{r}" for r in rels_all])
fit_ev = ols(long["y"], Xe, nme, cov="nonrobust")        # 상대시점 더미가 소수 관측만 덮어 HC1/군집 SE가 붕괴하므로 동분산 SE 사용(근사)
ev_all = fit_ev["table"].loc[[f"ev_{r}" for r in rels_all]].copy(); ev_all.index = rels_all
ev = ev_all.loc[[r for r in rels_all if -8 <= r <= 8]]                    # 그림은 -8..8만
fig, ax = plt.subplots(figsize=(9, 3.6)); ax.errorbar(ev.index, ev["coef"], yerr=1.96 * ev["se"], fmt="o")
ax.axhline(0, color="k"); ax.axvline(-0.5, color="gray", ls=":"); ax.set_title("이벤트 스터디(기준: 상대시점 -1)"); plt.show()
pre_ev = ev_all.loc[[r for r in rels_all if r < 0]]
wald = float((pre_ev["coef"] ** 2 / pre_ev["se"] ** 2).sum()); pw = 1 - stats.chi2.cdf(wald, len(pre_ev))
print(f"사전계수 근사 Wald(동분산 SE, 공분산 무시) = {wald:.2f}, p = {pw:.3f}  (작으면 평행추세 의심; 근사 검정임)")

## 6. 방법 비교

In [ ]:
rows = [{"방법": "ITS (HAC) 수준변화 β2", "추정": float(fit_its["beta"][2]), "se": float(fit_its["se"][2])},
        {"방법": "TWFE DID τ (군집 SE)", "추정": float(fit_did["table"].loc["D", "coef"]), "se": float(fit_did["table"].loc["D", "se"])}]
if "gap" in globals():
    rows.append({"방법": "SCM 사후 평균 gap", "추정": float(gap[T0:].mean()), "se": np.nan})
tab = pd.DataFrame(rows).set_index("방법")
if TAU is not None: tab["주입된 tau"] = TAU
show(tab)
print("ITS의 수준변화(β2)는 처치 직후 한 시점의 변화이고, DID·SCM은 사후 평균효과라 정의가 다르다. 비교 시 정의를 맞춰 읽는다.")

## 3. 결과 해석 작성
1. `DATA_KIND`를 먼저 적는다. 합성 패널이면 주입 효과 $\tau$와 추정치를 비교해 **추정기의 회복 여부**를 서술한다. 실제 자료(Prop 99)이면 추정치를 인과효과로 단정하지 않고 가정(평행추세·간섭 없음·도너 풀)과 함께 해석한다.
2. 이벤트 스터디의 사전계수와 사전 RMSPE가 **식별 가정의 증거**로 충분한지, 아니면 단지 "반증 실패"인지 구분한다.
3. 처치 단위가 하나(또는 소수)일 때 표준오차·p값의 신뢰 한계를 논의한다(순열 p의 최소값 $1/(1+J)$).
4. 한계: 동시 정책, 예상 효과(anticipation), 전염(spillover), 효과의 이질성(동시기 처치가 여러 개일 때 TWFE의 편의).

**제출:** 세 방법의 추정표, 이벤트 스터디/SCM gap 그림, 위약 결과, 가정별 점검 표, 해석 단락.